In [ ]:
%%capture
%env AWS_PROFILE=kmnlp

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import json
import os
from os.path import basename, dirname, splitext
from time import perf_counter

import fsspec
import numpy as np
import seaborn as sns
import s3fs
import xarray as xr
from dask.distributed import Client, LocalCluster
from fsspec.implementations.reference import ReferenceFileSystem
from matplotlib import pyplot as plt
from rich import print as rprint
from tqdm import tqdm

from llm_agent.dao.knowledge_graph_dao import knowledge_graph_dao

In [ ]:
%matplotlib inline
sns.reset_defaults()

In [ ]:
s3 = s3fs.S3FileSystem()

In [ ]:
cluster = LocalCluster()
client = cluster.get_client()

In [ ]:
datasets = knowledge_graph_dao.fetch_variables()
len(datasets)

In [ ]:
experiment_name = "zarr_parquet_zarrv3"
benchmarks_path = f"./data/{experiment_name}.json"
os.makedirs(dirname(benchmarks_path), exist_ok=True)
benchmarks_data = {}

In [ ]:
# with open(benchmarks_path, "r") as f:
#     benchmarks_data = json.load(f)

# ds_zarr_open_times = benchmarks_data["ds_zarr_open_times"]
# ds_zarr_file_bytes = benchmarks_data["ds_zarr_file_bytes"]
# ds_slice_max_info = benchmarks_data["ds_slice_max_info"]

---

In [ ]:
def load_zarr_ds(uri: str) -> xr.Dataset:
    return xr.open_dataset(
        uri,
        engine="kerchunk",
        storage_options={
            "remote_protocol": "s3",
            "target_protocol": "s3",
        },
    )

In [ ]:
def json_uri(parquet_uri):
    return f"{splitext(parquet_uri)[0]}.json"

In [ ]:
ds_zarr_open_times: dict[str, float] = {}
ds_xr: dict[str, xr.Dataset] = {}
with tqdm(datasets) as bar:
    for ds in bar:
        tic = perf_counter()
        ds_xr[ds.iri] = load_zarr_ds(ds.source)
        toc = perf_counter()
        ds_zarr_open_times[ds.iri] = toc - tic

In [ ]:
ds_zarr_file_bytes: dict[str, int] = {ds.iri: s3.stat(json_uri(ds.source))["size"] for ds in datasets}

In [ ]:
plt.close("all")
fig, ax = plt.subplots()
inds = np.arange(len(datasets), dtype=int)
x = np.array(list(ds_zarr_file_bytes.values()))
y = np.array(list(ds_zarr_open_times.values()))
ax.scatter(x, y, marker="o", fc="none", ec="steelblue", lw=1.5, alpha=1)
# mask = y > 30
# for i, _x, _y in zip(inds[mask], x[mask], y[mask]):
#     ax.annotate(
#         text=basename(datasets[i].iri), 
#         xy=(_x, _y), 
#         xytext=(-100, -15),
#         xycoords="data",
#         textcoords="offset points",
#         fontsize=9,
#     )
ax.set_xscale("log", base=2)
ax.set_xlabel("Zarr reference file size (bytes)")
ax.set_ylabel("Time (seconds)")
ax.set_title("Time to load into Xarray dataset")
plt.grid(ls="--", c="k", alpha=0.1)
plt.show()

---

## Computation time

In [ ]:
def get_reasonably_large_subset(da: xr.DataArray, *, min_size: int) -> xr.DataArray:
    t_lim = 1
    if da.nbytes < min_size:
        da_sliced = da
    else:
        da_sliced = da.isel(time=slice(0, t_lim))
        while da_sliced.nbytes < min_size:
            t_lim *= 2
            da_sliced = da.isel(time=slice(0, t_lim))
    return da_sliced

In [ ]:
ds_slice_max_info: dict[str, dict[str, int | float]] = {}
with tqdm(datasets) as bar:
    for ds in bar:
        da = ds_xr[ds.iri][ds.variables[0].field_name]
        da_sliced = get_reasonably_large_subset(da, min_size=10 * (2**20))
        tic = perf_counter()
        _ = da_sliced.mean().compute()
        toc = perf_counter()
        ds_slice_max_info[ds.iri] = {
            "bytes": da_sliced.nbytes,
            "time": toc - tic,
        }


In [ ]:
plt.close("all")
fig, ax = plt.subplots()
ds_names = np.array([basename(k) for k in ds_slice_max_info.keys()])
x = np.array([v["bytes"] for v in ds_slice_max_info.values()])
y = np.array([v["time"] for v in ds_slice_max_info.values()])
# mask = x > 2**20
# ds_names = ds_names[mask]
# x = x[mask]
# y = y[mask]
x = x / 2**20
r = y / x
sns.barplot(y=ds_names, x=r, orient="h", ax=ax)
ax.set_xlabel("Seconds per MB")
ax.grid(ls="--", c="k", alpha=0.1)
plt.show()

---

In [ ]:
benchmarks_data["ds_zarr_open_times"] = ds_zarr_open_times
benchmarks_data["ds_zarr_file_bytes"] = ds_zarr_file_bytes
benchmarks_data["ds_slice_max_info"] = ds_slice_max_info

In [ ]:
with open(benchmarks_path, "w") as f:
    json.dump(benchmarks_data, f, indent=2)